In [1]:
import sys
# sys.path.insert(0, '../../Utilities/')

# from jax.example_libraries import optimizers
import jax.numpy as jnp
import numpy as np
from jax import grad, jit, vmap, value_and_grad, hessian
from jax import random
from functools import partial
import flax.linen as nn
import jax
import optax
import pickle
from datetime import datetime
import os
import numpy as np
from scipy import sparse
import time
from jax import grad, hessian, jit, vmap, pmap
from jax.tree_util import tree_leaves
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import flax.linen as nn
import optax
import optax.tree_utils as otu

from minmax_opt_class_mgnn import MinMaxOptProb

In [2]:
eps0 = 1e-4
def smooth_sign(x, eps = eps0):
    return 2*jax.nn.sigmoid(x/eps)-1

# def u0(x):
#     return 0.5 * (smooth_sign(x - 0.4) - smooth_sign(x - 0.6))

# def u1(x):
#     return 0

# def u2(x):
#     return 0


# Define boundary and initial conditions
def u0(x):
    # return 2*smooth_sign(x) - 1
    return -0.5*(jnp.sign(x) - 1)
    # return -jnp.sin(jnp.pi*x)
def u_exact(tx):
    t, x = tx[0], tx[1]
    return u0(x-0.5*t)
vec_u_exact = vmap(u_exact)

# Boundary at x=-1
def u1(x):
    return 1

# Boundary at x=1
def u2(x):
    return 0



def res_visc(u_quad, u_t, u_x, u_xx, nu, u_exact=0):
    # print(f"u_quad shape: {u_quad.shape}")
    # print(f"u2_x shape: {u2_x.shape}")
    # print(f"u_xx shape: {u_xx.shape}")
    return u_t + u_quad*u_x - nu*u_xx
    # return u_t + 0.8*u_x - nu*u_xx
    # return jnp.zeros_like(u_quad)
    # return u_quad - u_exact

def res(u_quad, u_t, u_x):
    return u_t + u_quad * u_x
    # return u_t + 0.8*u_x

# Initialize separate loss history lists for each stage
pre_loss_int_history = []
pre_loss_bc_history = []
train_loss_int_history = []
train_loss_bc_history = []

# Initialize the optimization problem
optProb = MinMaxOptProb(
    N_t=16,
    N_x=32,
    lr_theta=1e-3,
    N_bc=100,
    N_ic=200,
    t_min=0, t_max=1, x_min=-1, x_max=1,
    hidden_dims_list=[(40, 40), (40, 40), (40, 40)],
    seed=0,
    u0=u0, u1=u1, u2=u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,    
)

In [3]:
optProb.train_process()


Starting Train Stage with MGDL


wandb: Currently logged in as: haoli1 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Stage 1: Sequential Grade Training

Training Grade 0


IndexError: Too many indices: 0-dimensional array indexed with 1 regular index.